# 01. Collectives and DDP Gradient Bucketing Mechanics

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Li et al. (VLDB 2020, *PyTorch Distributed: Experiences on Accelerating Data Parallel Training*), Patarasuk & Yuan (J. Parallel Distrib. Comput. 2009, *Bandwidth Optimal All-reduce on Dual-Ring*).
- **Purpose:** Implement Ring AllReduce collective primitives and contiguous gradient bucketing to overlap backward computation with inter-node gradient communication.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [PEFT Representation Finetuning Loreft](../15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/05_peft_representation_finetuning_loreft.ipynb) $\to$ `🔗 Downstream Syntheses:` [Device Mesh and DTensor Sharding Placements](02_device_mesh_and_dtensor_sharding_placements.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Standard DDP replicates the entire model across every GPU. When model parameters exceed single-GPU memory capacity ($>14\text{B}$ params on 80GB HBM), pure DDP causes OOM. Transition to FSDP2 / ZeRO-3 parameter sharding.

## Mathematical Foundations & Governing Equations

$$
T_{\text{ring}} = 2(W - 1)\alpha + 2 \frac{W - 1}{W} S \beta, \quad \nabla W_{\text{ddp}} = \frac{1}{W} \sum_{r=0}^{W-1} \nabla W_r
$$

### Asymptotic Time & Memory Complexity
- **Ring Communication Bandwidth:** $\mathcal{O}(2 S \beta)$ as $W \to \infty$, strictly independent of world size $W$.
- **Gradient Bucketing Buffer:** $\mathcal{O}(B_{\text{cap}})$ flat contiguous memory allocation (default 25 MB).

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def ring_allreduce_simulate(rank_data: torch.Tensor) -> torch.Tensor:
    world_size, total_elements = rank_data.shape
    chunk_size = total_elements // world_size
    chunks = rank_data.view(world_size, world_size, chunk_size).clone()

    for step in range(world_size - 1):
        send_chunks = chunks.clone()
        for r in range(world_size):
            recv_idx = (r - step - 1) % world_size
            prev_rank = (r - 1) % world_size
            chunks[r, recv_idx] += send_chunks[prev_rank, recv_idx]

    for step in range(world_size - 1):
        send_chunks = chunks.clone()
        for r in range(world_size):
            recv_idx = (r - step) % world_size
            prev_rank = (r - 1) % world_size
            chunks[r, recv_idx] = send_chunks[prev_rank, recv_idx]

    return chunks[0].view(total_elements)

def ddp_bucketed_allreduce(rank_grads: torch.Tensor, bucket_cap: int) -> tuple[torch.Tensor, list[torch.Tensor]]:
    world_size, num_params = rank_grads.shape
    num_buckets = (num_params + bucket_cap - 1) // bucket_cap
    bucket_results = []
    for b in range(num_buckets):
        start, end = b * bucket_cap, min((b + 1) * bucket_cap, num_params)
        reduced = torch.sum(rank_grads[:, start:end], dim=0) / float(world_size)
        bucket_results.append(reduced)
    return torch.cat(bucket_results, dim=0), bucket_results

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

ar_sum = ring_allreduce_simulate(weights["rank_inputs"])
assert float(torch.max(torch.abs(ar_sum - weights["allreduce_sum"]))) < 1e-5

ddp_grads, buckets = ddp_bucketed_allreduce(weights["rank_grads"], 8)
assert float(torch.max(torch.abs(ddp_grads - weights["ddp_grads_out"]))) < 1e-5
assert float(torch.max(torch.abs(buckets[0] - weights["bucket_1_reduced"]))) < 1e-5
assert float(torch.max(torch.abs(buckets[1] - weights["bucket_2_reduced"]))) < 1e-5
print("All Ring AllReduce & DDP Gradient Bucketing checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: ring_allreduce_simulate(weights["rank_inputs"]), iters=50, name="Ring AllReduce Simulation"),
    benchmark_fn(lambda: ddp_bucketed_allreduce(weights["rank_grads"], 8), iters=50, name="DDP Bucketed AllReduce"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Unbucketed Individual Param Reduction vs Contiguous Flat Bucketing
grads = weights["rank_grads"]
num_params = grads.shape[1]

# Simulating unbucketed individual collective dispatch overhead
# Each parameter tensor initiates a separate communication transaction (latency penalty alpha)
alpha_latency_us = 5.0  # NCCL launch / network barrier overhead in microseconds
unbucketed_overhead_us = num_params * alpha_latency_us

# Bucketed contiguous reduction reduces transaction count to ceil(num_params / bucket_size)
bucket_size = 8
num_buckets = (num_params + bucket_size - 1) // bucket_size
bucketed_overhead_us = num_buckets * alpha_latency_us

print(f"Unbucketed Latency Overhead ({num_params} individual collectives): {unbucketed_overhead_us:.1f} us")
print(f"Bucketed Latency Overhead   ({num_buckets} bucket collectives):     {bucketed_overhead_us:.1f} us")
assert unbucketed_overhead_us > bucketed_overhead_us, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Bandwidth Invariance:** Ring AllReduce transfers $2 \frac{W-1}{W} S$ bytes per node, maintaining near constant per-node communication volume regardless of cluster scale.
2. **Latency Amortization:** Packing parameter gradients into 25MB buckets amortizes collective launch overhead, saturating line-rate network bandwidth (>95% RoCE/InfiniBand).
3. **Compute-Comm Overlap:** Reversing bucket reduction to match autograd traversal allows communication of deep layers to execute concurrently with computation of shallow layers.

---
### Cross-Domain Generalisation Matrix
| Domain | Collective Communication Analogue | Practical Scaling Advantage |
| :--- | :--- | :--- |
| **Large-Scale Weather & Climate** | Halo exchange / domain boundary reduction | Overlaps physical boundary communication with grid CFD computation |
| **Federated Learning & Edge** | Secure multi-party aggregation (SMC) | Batches client gradient reports into encrypted buckets before consensus |
| **Distributed Graph Processing** | Bulk Synchronous Parallel (BSP) superstep | Amortizes inter-partition message exchange through contiguous vertex staging |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_collectives_and_ddp_gradient_bucketing_mechanics.md](../../markdown/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.md)
- **PyTorch 2.x Module:** [01_collectives_and_ddp_gradient_bucketing_mechanics.py](../../pytorch2/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.py)
- **Functional JAX Module:** [01_collectives_and_ddp_gradient_bucketing_mechanics.py](../../jax/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.py)
- **Native Rust Candle Track:** [01_collectives_and_ddp_gradient_bucketing_mechanics.rs](../../candle/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.rs)
- **Hardware Kernel (CPU / WGSL):** [01_collectives_and_ddp_gradient_bucketing_mechanics.rs](../../kernels/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.rs)
- **Golden SafeTensors:** [01_collectives_and_ddp_gradient_bucketing_mechanics.safetensors](../../weights/16_distributed_and_scalable/01_collectives_and_ddp_gradient_bucketing_mechanics.safetensors)

---
**Curriculum Navigation:**  
&larr; Previous: — | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 02. Device Mesh and DTensor Sharding Placements](02_device_mesh_and_dtensor_sharding_placements.ipynb) &rarr;